In [ ]:
%pip install pandas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.arima_process import ArmaProcess

# Time Series
## ARIMA 

### **AR**IMA - авторегрессия

Уравнение модели авторегрессии (AR)
$$ x_t = const + \sum_{i=1}^p\beta_ix_{t-i} + \epsilon_t$$

### AR**I**MA - дифференцирование/интегрирование

В самом начале берется разность n порядка (Дифференцирование)

В самом конце высчитывается обратная операция (Интегрирование)

### ARI**MA** - (Moving average)

Используется метод максимального правдоподобия MLE

Правдоподобие:
$$L(\phi, \theta, c, \sigma^2) = \prod_{t=1}^{T} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{\varepsilon_t^2}{2\sigma^2}\right)$$


Гиперпараметры:
| p | d | q |
| :-: | :-: | :-: |
| Количество предыдущих значений, участвующих в AR модели | Какое количество раз надо взять разность между соседними точками, чтобы "убрать тренд", тем самым сделав ряд стационарным | Количество прошлых ошибок, которые надо помнить |


## - fit


Сначала дифференцируем d раз

Затем AR и MA начинают строить модель
- AR - строит прогноз, основываясь на prev значениях ряда
- MA - уточняет прогноз, опираясь на ошибки модели

Уравнение ARIMA:
$$\tilde{X}_t = c + \underbrace{\phi_1 \tilde{X}_{t-1} + \dots + \phi_p \tilde{X}_{t-p}}_{\text{AR-часть}} + \underbrace{\theta_1 \varepsilon_{t-1} + \dots + \theta_q \varepsilon_{t-q}}_{\text{MA-часть}} + \varepsilon_t$$

Если для $$X_{t-i}$$ существует соответствующее ему реальное значение, берется оно, иначе берется уже "предсказанное значение" (Чтобы рассчитывать прогноз далее одного значения)

В конце интегрируем d раз, чтобы результат соответствовал по пространству ряда, поданному на вход модели

Каждую итерацию обновляются коэффициенты
Модель заканчивает итерации обновления коэффициентов, если:
- Достигнута сходимость (Разница между новыми и предыдущими коэффициентами стремится к нулю)
- прирост правдоподобия, что модель описывает данные, стал нулевым
- Превышен лимит итераций

## - inference - 

Модель обладает рассчитанными коэффициентами для AR и MA, в уравнение ARIMA подставляются признаки, возвращается предсказанное значение целевой переменной

### 4.8        (Time Series)  A classic dataset used in time series lectures is the quarterly earnings per share (EPS) for Johnson & Johnson stock.  Shumway and Stoffer [2006] utilize this dataset in their text and provide a copy of the dataset.   

### There are 84 values in this dataset covering the EPS from the first quarter of 1960 to the fourth quarter of 1980.

In [ ]:
# Смотрю, как вообще устроены данные
!head resources/8-lab/jj.dat
!tail resources/8-lab/jj.dat

In [ ]:
df = pd.read_csv('resources/8-lab/jj.dat', header=None).astype(np.float32)
df.columns = ['EPS']
df['time'] = pd.DataFrame([i for i in range(84)]).astype(np.int8)
df


##  a.      After obtaining a copy of the dataset, plot the quarterly EPS vs. time.  Describe any patterns that you observe.

Паттерны:
- Долгосрочный восходящий тренд - с каждым годом акция всё больше растёт
- Яркая сезонность - В рамках одного года (4 точки соседние) очень хороший разброс, особенно в последние годы
- Растущая амплитуда колебаний - с каждым годом ```max(EPS)-min(EPS)``` всё больше

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(df['time'], df['EPS'], marker='o', color='orange', markersize=4)
plt.xlabel('Time')
plt.ylabel('EPS')
plt.grid()

plt.show()

## b.      In order to perform an ARIMA model, the time series will need to be transformed to remove any trend.  Plot the difference of xt and xt-1, for all t > 0.   Has this difference adequately detrended the series? Does the variability of the EPS appear constant over time?

- Взятие разности помогло устранить восходящий тренд
- Дисперсия не является постоянной: она растёт со временем

### Why does the constant variance matter?

Метод максимального правдоподобия требует константной дисперсии
$$f(\varepsilon_t) = \frac{1}{\sqrt{2\pi \sigma^2}} \exp\left( -\frac{\varepsilon_t^2}{2\sigma^2} \right)$$
$$L = f(\varepsilon_1) \cdot f(\varepsilon_2) \cdot \dots \cdot f(\varepsilon_T)$$
Если дисперсия не будет постоянной, то для каждого $$f(\epsilon_i)$$ будет своя $$\sigma_i$$, что повлечет дополнительные степени свободы, на разрешение которых не хватит точек во временном ряду (Следовтельно система неразрешима)
Потому алгоритм и написан для "постоянной дисперсии", на уровне отработки ```ARIMA.fit()``` просто результат будет неправильным, тк мы заменим p сигм на какую-то одну 

In [ ]:
diffs = df['EPS'].diff()

plt.figure(figsize=(12, 6))

plt.plot(df['time'], diffs, marker='o', color='green', markersize=4)
plt.xlabel('Time')
plt.ylabel('EPS')
plt.grid()

plt.show()

## c.       Plot the log10 of the quarterly EPS vs. time and plot the difference of log10(xt ) and log10(xt-1) for all t > 0

In [ ]:
log_ = np.log10(df['EPS'])
log_diffs = log_.diff()

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

axes[0].plot(df['time'], log_, marker='o', color='orange', markersize=4)
axes[0].set_title('Log10(EPS) vs Time')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Log10(EPS)')
axes[0].grid()

axes[1].plot(df['time'], log_diffs, marker='o', color='green', markersize=4)
axes[1].set_title('Log10(EPS).diff() vs Time ')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Log10(EPS).diff()')
axes[1].grid()

plt.show()

###  Has this adequately detrended the series?
Да, взятие разности первого порядка устранило тренд

### Has the variability of the differenced log10(EPS) become more constant? 
Дисперсия стала более постоянной, причем можно заметить, что можно разделить её на 3 части\
от 0 до 20; от 20 до 60; от 60 до 84

## d.      Treating the differenced log10 of the EPS series as a stationary series, plot the ACF and PACF of this series.  What possible ARIMA models would you consider and why?

- ```ACF``` - Автокорреляционная функция. Измеряет полную корреляцию между текущим моментом времени i и прошлым моментом времени i - k, где k - лаг (Какой шаг назад во времени делаем)
 $$\rho_k = \frac{\operatorname{Cov}(y_t,\, y_{t-k})}{\operatorname{Var}(y_t)}$$
Включает косвенное влияние промежуточных значений
- ```PACF``` - Частная автокорреляционная функция. Измеряет чистую корреляцию между текущим моментом времени i и прошлым моментом времени i - k, где k - лаг (Какой шаг назад во времени делаем)
  $$\varphi_{kk} = \operatorname{Corr}\!\left(y_t - \hat{y}_t,\; y_{t-k} - \hat{y}_{t-k}\right)$$
Корреляция $$y_t \ и \  y_{t-k}$$ после того как линейно убрано влияние промежуточных значений



In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# ACF
plot_acf(log_diffs.dropna(), ax=axes[0], lags=40)
axes[0].set_title('Autocorrelation Function')

# PACF
plot_pacf(log_diffs.dropna(), ax=axes[1], lags=40)
axes[1].set_title('Partial Autocorrelation Function')

plt.tight_layout()
plt.show()

Основываясь на графике PACF, до третьего включительно лаг выходит за границы доверительного интервала, в связи с чем мы понимаем, что дальше 3 лага нет смысла включать в модель, тк это будет случайное возмущение\
=> p = 3


Основываясь на графике разности первого порядка с предыдущего пункта, мы понимаем, что её достаточно для удаления тренда\
=> d = 1


Основываясь на графике ACF, функция затухает плавно (А не резко), что означает, что это чистая авторегрессия\
=> q = 0\
Также для сравнения возьмем q = 1 и q = 4


## e.      Run the proposed ARIMA models from part d and compare the results.  Identify an appropriate model.  Justify your choice.  

- ```AIC``` - Информационный критерий Акаике. Ищет модель, дающую наиболее лучший прогноз
  $$\text{AIC} = 2k - 2\ln(L)$$
- ```BIC``` - Байесовский информационный критерий. Ищет модель, с более сбалансированным соотношением точность/коэффициенты
 $$\text{BIC} = k\ln(n) - 2\ln(L)$$

L - likehood
k - количество параметров (Коэфф авторегрессии + коэффициенты скользящего среднего + дисперсия)

In [ ]:
model_configs = [
    (3, 1, 0), # Подобрано в пункте d
    (1, 1, 1), # Базовая конфигурация
    (3, 1, 1), # Подобрано в пункте d + предположение, что MA зависит от 1 предыдущего
    (3, 1, 4), # Подобрано в пункте d + предположение, что MA зависит от 4 предыдущих
]

res = []
fitted_models = {}

for i in model_configs:
    arima = ARIMA(log_, order=i)
    fitted = arima.fit() 
    fitted_models[i] = fitted

    res.append({
        '(p, d, q)' : str(i),
        'AIC' : fitted.aic,
        'BIC' : fitted.bic,
        'Likehood' : fitted.llf,
        'Sigma**2' : fitted.params.get('sigma2', None)
    })

df_ = pd.DataFrame(res).sort_values(by='AIC')
display(df_)

Основываясь на сравнении наборов гиперпараметров, (3, 1, 4) показал себя лучше всего по всем 4 метрикам

## 4.9        (Time Series)  Why is the choice of natural log or log base 10 in Problem 4.8 somewhat irrelevant to the transformation and the analysis?

Можно выбрать произвольное основание для логарифма, поскольку
$$log_{основание}(a) = \frac{ln(a)}{ln(основание)} = \frac{1}{ln(основание)} \cdot ln(a)$$

Всегда можно привести к желаемому основанию, а домножение всей модели на константу не повлечет изменений в логике работы модели

## 4.10    (Time Series)  Why is the value of the ACF for lag 0 equal to one?
### Note: The ACF R function plots the value at lag 0. 

Значение автокорреляционной функции для нулевого лага всегда равняется одному по причине того, что измеряется корреляция относительно самой себя

## 4.11    (Time Series) Using arima.sim in R, simulate 10,000 points for AR(p) p = 1,2,3,4.
## Plot the simulated series, ACF and PACF.  What pattern emerges between p and the plots? 

(0.9, -0.5, .2, -.3) # Коэффициенты

### The choice of the coefficients needs to be carefully made to ensure that the simulated series is a stationary series.   Suggestion: Simply drop off the last coefficient to get to the AR(3) inputs, then the AR(2) inputs, and finally the AR(1) inputs.


In [ ]:
np.random.seed(42)
n = 10_000

ARs = { # ключом является порядок ряда
    1: [1, -0.9],
    2: [1, -0.9, 0.5],
    3: [1, -0.9, 0.5, -0.2],
    4: [1, -0.9, 0.5, -0.2, 0.3]
}

fig, axes = plt.subplots(4, 2, figsize=(14, 16))

for p, arCfs in ARs.items():
    maCfs = [1]

    process = ArmaProcess(ar=arCfs, ma = maCfs)
    series =  process.generate_sample(nsample = n)

    # ACF
    plot_acf(series, ax=axes[p - 1, 0], lags=15, title=f"ACF")
    axes[p - 1, 0].set_ylabel(f"p = {p}")
    
    # PACF
    plot_pacf(series, ax=axes[p - 1, 1], lags=15, title=f"PACF")

plt.show()

1 порядок:
- PACF - все лаги, после первого, находятся в доверительном интервале


2 порядок:
- PACF - все лаги, после второго, находятся в доверительном интервале


2 порядок:
- PACF - все лаги, после третьего, находятся в доверительном интервале


3 порядок:
- PACF - все лаги, после четвертого, находятся в доверительном интервале


ACF сходится к нулю, что свидетельствует о стационарности временных рядов


## 4.12    (Time Series)  Using arima.sim in R, simulate 10,000 points for MA(p) p = 1,2,3,4.
## Plot the simulated series, ACF and PACF.  What pattern emerges between p and the plots?

(-1.9, 1.7, -1.5, 1.5) # Коэффициенты

### The choice of the coefficients needs to be carefully made to ensure that the simulated series is a stationary series.   Suggestion: simply drop off the last coefficient to get to the MA(3) inputs, then the MA(2) inputs, and finally the MA(1) inputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

np.random.seed(42)
n = 10_000

MAs = { # ключом является порядок ряда
    1: [1, -1.9],
    2: [1, -1.9, 1.7],
    3: [1, -1.9, 1.7, -1.5],
    4: [1, -1.9, 1.7, -1.5, 1.5]
}

fig, axes = plt.subplots(4, 2, figsize=(14, 16))

for p, maCfs in MAs.items():
    ar_cfs = [1]

    process = ArmaProcess(ar=ar_cfs, ma=maCfs)
    series = process.generate_sample(nsample=n)

    # ACF
    plot_acf(series, ax=axes[p - 1, 0], lags=15, title=f"ACF")
    axes[p - 1, 0].set_ylabel(f"p = {p}")
    
    # PACF
    plot_pacf(series, ax=axes[p - 1, 1], lags=15, title=f"PACF")

plt.show()

Чистый MA процесс. ACF помогает определить порядок MA (Определить насколько шагов назад ошибка всё еще имеет влияние свыше случайного возмущения)